# Demo: numerical LP vs analytical criteria

We classify a 2D grid of probability vectors as classical or nonclassical
using (a) the LP solver and (b) the analytical criteria, then overlay the
decision boundaries. All computation is delegated to `src`.

In [ ]:
# Make the project importable when running from the notebooks/ directory.
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.config import apply_thread_limits
apply_thread_limits()

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.comparison.scan_2d import run2D
from src.config import SCAN


In [ ]:
from src.config import PATHS

def load_scan_archive(path):
    """Read a run2D .npy archive back into its components."""
    with open(path, 'rb') as f:
        PA_all = np.load(f)
        PB_all = np.load(f)
        Prest = np.load(f)
        boundaries = np.load(f, allow_pickle=True)
        boundaries_ana = np.load(f, allow_pickle=True)
        boundaries_anaK = np.load(f, allow_pickle=True)
        boundaries_anaM = np.load(f, allow_pickle=True)
        nonclass_num = np.load(f)
        nonclass_ana = np.load(f)
    return dict(PA_all=PA_all, PB_all=PB_all, Prest=Prest,
                boundaries=boundaries, boundaries_ana=boundaries_ana,
                boundaries_anaK=boundaries_anaK, boundaries_anaM=boundaries_anaM,
                nonclass_num=nonclass_num, nonclass_ana=nonclass_ana)

# Demo: the linear-programming nonclassicality solver

This notebook demonstrates the primary solver implemented in `src.solvers.lp_solver`.
The notebook itself contains only presentation and visualization code; all numerical routines are imported directly from `src`.

## Problem

Given a finite set of occupation probabilities, we test whether they are compatible with a classical state, i.e., with a mixture of Poisson distributions,

$$
P_k = \int \mathrm d\mu\, \rho(\mu)\, p_k(\mu),
\qquad
\rho(\mu)\ge 0,
$$

where

$$
p_k(\mu)=e^{-\mu}\frac{\mu^k}{k!}.
$$

Any incompatibility with this structure certifies nonclassicality.

To test this, we use the witness

$$
f(\boldsymbol\lambda)
=
\max\!\Bigl(
0,
\max_{\mu\ge0}
\boldsymbol\lambda\cdot\mathbf p(\mu)
\Bigr)
-
\boldsymbol\lambda\cdot\mathbf P.
$$

Minimizing $f$ over $\boldsymbol\lambda$ leads to a semi-infinite linear program: there are finitely many optimization variables but infinitely many constraints, one for every $\mu\ge0$.

The reported quantity

$$
\mathrm{num}
=
-\min_{\boldsymbol\lambda}f
$$

is positive whenever the supplied occupation probabilities certify nonclassicality in exact arithmetic. In a numerical scan we use an explicit threshold: a grid point is marked nonclassical only when `num > bias`. The default is `SCAN.bias = 1e-8`.

You can change this threshold for one calculation by passing, for example, `bias=1e-10` or `bias=1e-6` to `run2D(...)` / `analyze_2d(...)`. To change the project default, edit `scan.bias` in `config.yaml` before importing `src`, or point `NONCLASSICALITY_CONFIG` to another YAML file. Smaller `bias` values are more sensitive but less conservative close to numerical precision; larger values are more conservative.

The analytical criteria are different: in the current implementation they do not have a separate configurable threshold. They classify by the sign of the analytical violation, for example `violation > 0` or `violation < 0`, after floating-point evaluation. Values extremely close to zero should therefore still be interpreted with care, especially when measured or computed probabilities are very small.

---

## Majorization criteria

The majorization criteria can become computationally demanding when many probabilities are included.

The precomputed archives stored in

```text
data/all_genK/
```

contain all majorization witnesses up to $N_P=9$:

| $N_P$ | Number of witnesses |
|---------|--------------------:|
| 4 | 7 |
| 5 | 55 |
| 6 | 489 |
| 7 | 4,555 |
| 8 | 45,998 |
| 9 | 479,398 |

No archives are currently provided beyond $N_P=9$.

For large probability sets it is often preferable to disable the majorization pass:

```python
run2D(..., use_analytical_m=False)
```

In this notebook the analytical criteria are included mainly for comparison. Any state certified by the analytical witnesses is also certified by the numerical LP criterion, which represents the most general witness considered here. However, analytical criteria can still be useful as quick diagnostics, and they may remain informative when selected probabilities are close to the numerical precision scale.

---

## Why are the scan bounds nonzero?

The lower scan bounds `dPA` and `dPB` are chosen to be small but nonzero.

This avoids points lying exactly on the boundaries of the probability simplex, where some probabilities vanish. Such points may lead to ill-defined analytical witnesses and can occasionally produce numerical instabilities during LP optimization or boundary extraction.


In [ ]:
fi = '../results/demo_probability_plane.npy'
print('scan decision threshold bias =', SCAN.bias)
boundaries, boundaries_ana, boundaries_anaK, boundaries_anaM = run2D(
    N0=120, N1=120,
    indx=np.array([0, 1, 2, 3], dtype=np.int16),
    dPA=1e-10, uPA=0.9, dPB=1e-10, uPB=0.3,
    Prest=np.array([0.05, 0.05]),
    fi=fi,
)

## Overlay the decision boundaries

The thick orange curve is the (exact) LP boundary; the analytical criteria
are necessary but not sufficient, so they lie inside the LP region.

In [ ]:
plt.figure(figsize=(6, 5))
for b in boundaries:
    plt.plot(b[:, 0], b[:, 1], lw=6, color='black', label='LP')
for b in boundaries_anaK:
    plt.plot(b[:, 0], b[:, 1], "--", lw=4, color='red', label='analytical K')
for b in boundaries_anaM:
    plt.plot(b[:, 0], b[:, 1],  ":", lw=3, color='blue', label='analytical M')

handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())
plt.xlabel(r'$P_0$'); plt.ylabel(r'$P_1$')
plt.title('Nonclassicality boundaries'); plt.tight_layout(); plt.show()

The above routine stores, for each calculated point, whether nonclassicality is detected by the numerical and analytical criteria. The resulting data can be used to construct nonclassicality maps.

In [ ]:
data = load_scan_archive(fi)
print('grid:', data['nonclass_num'].shape, '| LP curves:', len(data['boundaries']))

In [ ]:
plt.figure(figsize=(6, 5))
plt.pcolormesh(data['PA_all'], data['PB_all'],
               data['nonclass_num'].astype(int) + data['nonclass_ana'].astype(int),
               shading='auto', cmap='viridis')
plt.colorbar(label='0 "classical" / 1 analytical-only / 2 both')
for b in data['boundaries']:
    plt.plot(b[:, 0], b[:, 1], lw=2, color='red')
plt.xlabel(r'$P_0$'); plt.ylabel(r'$P_1$')
plt.title('Sample nonclassicality map'); plt.tight_layout(); plt.show()

### Second example for $P_1$ and $P_3$ plane and fixed $P_2$, $P_5$, and $P_6$

In [ ]:
fi = '../results/demo_probability_plane.npy'
boundaries, boundaries_ana, boundaries_anaK, boundaries_anaM = run2D(
    N0=200, N1=200,
    indx=np.array([1, 3, 2, 5, 6], dtype=np.int16),
    dPA=1e-10, uPA=0.3, dPB=1e-10, uPB=0.04,
    Prest=np.array([0.05, 0.001,0.001]),
    fi=fi,
)

In [ ]:
plt.figure(figsize=(6, 5))
for b in boundaries:
    plt.plot(b[:, 0], b[:, 1], lw=6, color='black', label='LP')
for b in boundaries_anaK:
    plt.plot(b[:, 0], b[:, 1], "--", lw=4, color='red', label='analytical K')
for b in boundaries_anaM:
    plt.plot(b[:, 0], b[:, 1],  ":", lw=3, color='blue', label='analytical M')

handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())
plt.xlabel(r'$P_1$'); plt.ylabel(r'$P_3$')
plt.title(r'$P_2=0.05,\, P_5=0.001,\,P_6=0.001$'); plt.tight_layout(); plt.show()

In [ ]:
data = load_scan_archive(fi)
print('grid:', data['nonclass_num'].shape, '| LP curves:', len(data['boundaries']))
plt.figure(figsize=(6, 5))
plt.pcolormesh(data['PA_all'], data['PB_all'],
               data['nonclass_num'].astype(int) + data['nonclass_ana'].astype(int),
               shading='auto', cmap='viridis')
plt.colorbar(label='0 "classical" / 1 analytical-only / 2 both')
for b in data['boundaries']:
    plt.plot(b[:, 0], b[:, 1], lw=2, color='red')
plt.xlabel(r'$P_1$'); plt.ylabel(r'$P_3$')
plt.title(r'$P_2=0.05,\, P_5=0.001,\,P_6=0.001$'); plt.tight_layout(); plt.show()

## Further examples

Additional notebooks demonstrate:

- nonclassicality maps in the Jaynes--Cummings model,
- probabilistic boson addition,
- high-precision LP settings,
- CSV import/export,
- and diagnostic tools for probability-space scans.

See:

```text
notebooks/demo_2_analytical_vs_numerical.ipynb
notebooks/demo_3_jaynes_cummings.ipynb
notebooks/demo_4_probabilistic_boson_addition.ipynb
```